# Procedural Level Generation with Cellular Automata
## Capstone Project Notebook

Cellular Automata (CA) are discrete mathematical models widely used across various disciplines to simulate complex systems and emergent behaviours. A cellular automaton consists of a regular grid of cells, each in one of a finite number of states (e.g., "alive" or "dead"). The grid evolves over discrete time steps according to a set of predefined rules, which determine the next state of each cell based on its current state and the states of its neighbours. These simple rules can give rise to intricate and unpredictable patterns, making CA a powerful tool for modelling phenomena in physics, biology, computer science, and beyond.

Historically, cellular automata were introduced by John von Neumann and Stanislaw Ulam in the 1940s as a framework for studying self-replicating systems. The concept gained widespread recognition with John Conway's "Game of Life" in 1970, a two-dimensional CA that demonstrated how basic rules could produce complex, life-like behaviours. Today, CA are applied in diverse areas, such as simulating fluid dynamics, modelling tumour growth, and generating procedural content in computational design.

### CA in Game Development

In the context of game development, cellular automata are particularly valuable for **procedural level design**. They enable developers to generate diverse and organic game environments — such as terrains, caves, or dungeons — algorithmically, rather than crafting each element manually. This approach not only saves time but also enhances replayability by creating unique levels for each playthrough.

Several types of CA are commonly adapted for game level design:
- **Cave Generation Automata** — rules that simulate erosion or geological processes, creating natural-looking cave systems with open spaces and tunnels
- **Terrain Generation Automata** — designed to mimic natural landscape formation (mountains, rivers), widely used in open-world games
- **Dungeon Generation Automata** — tailored for structured yet randomised room-and-corridor layouts for roguelike games

Each type can be customised by adjusting grid size, neighbourhood definitions (Moore or von Neumann), and transition rules to suit a game's aesthetic and functional needs.

### Why CA for Level Design?

- **Procedural generation** — CA create unique levels each time, increasing replayability and reducing predictability
- **Efficiency** — automating level creation saves development time, allowing focus on gameplay mechanics or narrative
- **Organic complexity** — simple rules yield intricate, natural-looking structures (jagged caves, sprawling forests) that enhance immersion
- **Flexibility** — rules and parameters can be tweaked to match different genres, from survival games to strategy titles

### Mathematical Formulation

A cellular automaton is formally defined by:
- **Grid:** A lattice (typically 2D) of cells, each with a state from a finite set (e.g., $\{0, 1\}$ for empty or filled)
- **Neighbourhood:** A set of adjacent cells influencing a given cell, such as the Moore neighbourhood (8 surrounding cells) or von Neumann neighbourhood (4 cardinal directions)
- **Transition Rules:** A function mapping the current state of a cell and its neighbours to its next state

For a cell at position $(x, y)$ in grid $G_t$ at time $t$:

$$G_{t+1}(x, y) = f(G_t(x, y),\; N(x, y))$$

where $N(x, y)$ is the neighbourhood state count and $f$ is the rule function. Iterating this process over multiple steps refines the level structure.

### In this notebook you will:
1. Understand how **cellular automata** can generate natural-looking terrain
2. Implement the **cave generation algorithm** step by step
3. Analyse the effect of parameters on the generated levels
4. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for grid manipulation and Matplotlib for visualisation. We define two cell states — `FLOOR` (0, open space) and `WALL` (1, solid) — and a dark/light colour map that produces a clear dungeon-like aesthetic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
FLOOR = 0
WALL  = 1
CMAP = ListedColormap(['#2c3e50', '#ecf0f1'])

---
## 1 · The Cave Generation Algorithm

A simple cave-generation CA uses a binary grid ($0$ = empty/floor, $1$ = wall):
- If a **wall** cell has fewer than `death_limit` wall neighbours, it becomes empty (erosion)
- If an **empty** cell has more than `birth_limit` wall neighbours, it becomes a wall (sedimentation)

### Algorithm
1. **Initialise** a grid randomly: each cell is a wall with probability $p$ (typically 0.45–0.55)
2. **Iterate** CA rules for several steps:
   - Count **wall neighbours** in the 3×3 Moore neighbourhood (8 surrounding cells)
   - Wall with fewer than `death_limit` wall neighbours → floor
   - Floor with more than `birth_limit` wall neighbours → wall
3. **Seal borders**: force edge cells to be walls

| Parameter | Typical Value | Effect |
|-----------|--------------|--------|
| `initial_wall_chance` | 0.45–0.55 | Higher → denser walls |
| `death_limit` | 3–4 | Higher → walls die more easily (more open caves) |
| `birth_limit` | 4–5 | Higher → harder for new walls to form |
| `num_steps` | 3–7 | More steps → smoother caves |

No implementation example is provided in the project specification — developing and tuning the algorithm is the main task. The cells below give you a working baseline to build upon.

### Step 1: Initialise a random grid

The first step of the cave generation algorithm is to fill the grid with random noise. Each cell is independently set to `WALL` with probability `wall_chance` (typically around 0.50). This produces a salt-and-pepper pattern with no spatial structure — the CA rules will then iteratively smooth this noise into coherent cave-like regions.

The `seed` parameter ensures reproducibility: the same seed always produces the same initial grid, which is essential for debugging and for comparing the effect of different CA parameters on the same starting configuration.

In [ ]:
def init_grid(width, height, wall_chance=0.50, seed=42):
    rng = np.random.default_rng(seed)
    return (rng.random((height, width)) < wall_chance).astype(int)

test_grid = init_grid(100, 100)
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(test_grid, cmap=CMAP, vmin=0, vmax=1)
ax.set_title(f'Initial Random Noise (wall fraction: {np.mean(test_grid):.2f})', fontweight='bold'); ax.axis('off')
plt.tight_layout(); plt.show()

### Step 2: Count wall neighbours

The neighbourhood function is the heart of any CA. For each cell we count how many of its 8 Moore neighbours are walls. Cells outside the grid boundary are counted as walls — this ensures that the cave edges remain solid and prevents open spaces from "leaking" to the border.

This boundary convention has a significant effect on the generated levels: it biases edge cells towards remaining walls, naturally producing enclosed cave systems without needing an explicit border-sealing step during the rule application (though we still seal borders explicitly for safety).

In [ ]:
def count_wall_neighbours(grid, row, col):
    rows, cols = grid.shape
    count = 0
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr, nc = row + dr, col + dc
            if nr < 0 or nr >= rows or nc < 0 or nc >= cols:
                count += 1
            elif grid[nr, nc] == WALL:
                count += 1
    return count

### Step 3: One CA step

The `ca_step` function applies the cave generation rules to every interior cell simultaneously:

- A **wall** cell with fewer than `death_limit` wall neighbours becomes **floor** — this models erosion, carving out open spaces where walls are isolated.
- A **floor** cell with more than `birth_limit` wall neighbours becomes **wall** — this models sedimentation, filling in small gaps surrounded by solid rock.

Interior cells (rows 1 to $H-2$, columns 1 to $W-1$) are updated into a copy of the grid, and then the border cells are forced to `WALL` to keep the cave enclosed. The simultaneous update (writing to `new_grid` while reading from `grid`) prevents the scan order from biasing the result.

In [ ]:
def ca_step(grid, death_limit=4, birth_limit=5):
    rows, cols = grid.shape
    new_grid = grid.copy()
    for r in range(1, rows - 1):
        for c in range(1, cols - 1):
            walls = count_wall_neighbours(grid, r, c)
            if grid[r, c] == WALL:
                if walls < death_limit:
                    new_grid[r, c] = FLOOR
            else:
                if walls > birth_limit:
                    new_grid[r, c] = WALL
    new_grid[0, :] = WALL; new_grid[-1, :] = WALL
    new_grid[:, 0] = WALL; new_grid[:, -1] = WALL
    return new_grid

### Step 4: Full generation pipeline

The `generate_cave` function chains all the components: initialise a random grid, then apply the CA rules for `num_steps` iterations. It returns the final cave grid and the full history of intermediate grids (useful for visualising how the cave evolves from noise to structure).

After generation we print the open-space fraction — a quick sanity check. Typical caves have 40–60% open space depending on the initial wall chance and rule parameters.

In [ ]:
def generate_cave(width=100, height=100, wall_chance=0.50,
                   num_steps=5, death_limit=4, birth_limit=5, seed=42):
    grid = init_grid(width, height, wall_chance, seed)
    history = [grid.copy()]
    for _ in range(num_steps):
        grid = ca_step(grid, death_limit, birth_limit)
        history.append(grid.copy())
    return grid, history

cave, history = generate_cave()

---
## 2 · Visualising the Evolution

The sequence below shows how the CA transforms random noise into a structured cave over successive iterations. At step 0, the grid is pure noise. After the first step, small clusters begin to form as isolated wall/floor cells are absorbed by their majority neighbours. By step 3–5, coherent rooms and corridors have emerged — the level is "playable."

This noise-to-structure transformation is the fundamental trick behind CA-based procedural generation: **complex spatial patterns from simple local rules applied iteratively**.

In [ ]:
fig, axes = plt.subplots(1, len(history), figsize=(3.2 * len(history), 3.2))
for ax, (step, snap) in zip(axes, enumerate(history)):
    ax.imshow(snap, cmap=CMAP, vmin=0, vmax=1)
    ax.set_title('Noise' if step == 0 else f'Step {step}', fontsize=10)
    ax.axis('off')
fig.suptitle('Cave Generation: From Noise to Structure', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

---
## 3 · Parameter Exploration

The cave generator has three main knobs to turn: the **initial wall chance** $p$, the **death/birth limits**, and the **random seed**. Below we systematically vary each one to build intuition for how they shape the output.

First, we sweep the initial wall chance. Lower $p$ produces sparser initial noise, which leads to more open caves. Higher $p$ starts with denser noise, yielding narrower passages and more wall mass. The CA smoothing tends to amplify these initial biases.

Next we vary the CA rule parameters — `death_limit` and `birth_limit`. These control the balance between erosion and sedimentation. Low death limits preserve more walls; high birth limits make it harder for new walls to form, producing more open spaces. The six combinations below span a range from tight, maze-like caves to wide-open caverns.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, wc in zip(axes, [0.40, 0.48, 0.54, 0.62]):
    c, _ = generate_cave(wall_chance=wc, seed=42)
    ax.imshow(c, cmap=CMAP, vmin=0, vmax=1)
    ax.set_title(f'p={wc}  ({(1-np.mean(c))*100:.0f}% open)', fontsize=10)
    ax.axis('off')
fig.suptitle('Effect of Initial Wall Chance', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

Finally, we verify that the same parameters with different random seeds produce structurally different but stylistically consistent caves. This is the key value proposition of procedural generation: **infinite variety within a controlled aesthetic**.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (dl, bl) in zip(axes.flat, [(3,4),(4,4),(4,5),(4,6),(5,5),(3,6)]):
    c, _ = generate_cave(death_limit=dl, birth_limit=bl, seed=42)
    ax.imshow(c, cmap=CMAP, vmin=0, vmax=1)
    ax.set_title(f'death={dl}, birth={bl}  ({(1-np.mean(c))*100:.0f}%)', fontsize=10)
    ax.axis('off')
fig.suptitle('Effect of CA Rule Parameters', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, s in zip(axes, [1, 7, 42, 99]):
    c, _ = generate_cave(seed=s)
    ax.imshow(c, cmap=CMAP, vmin=0, vmax=1)
    ax.set_title(f'seed={s}', fontsize=10); ax.axis('off')
fig.suptitle('Same Parameters, Different Seeds', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

---
## 4 · Open Space Statistics

Beyond visual inspection, we can track a quantitative metric — the **percentage of open (floor) space** — as the CA iterates. This reveals how quickly the algorithm converges: after a few steps the open-space fraction stabilises, indicating that further iterations produce only minor cosmetic changes. Different initial wall chances converge to different steady-state openness levels, giving you a predictable knob for controlling level density.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for wc in [0.42, 0.48, 0.52, 0.58]:
    _, hist = generate_cave(wall_chance=wc, num_steps=10, seed=42)
    ax.plot([(1-np.mean(g))*100 for g in hist], lw=2, marker='o', markersize=4, label=f'p={wc}')
ax.set_xlabel('CA Step'); ax.set_ylabel('Open Space (%)')
ax.set_title('Open Space Convergence', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5 · Your Tasks

Implement the core CA level generator (done above) and add **at least two** features:

### Task A: Landscape Tuning
Use different rule sets to generate caves, lakes, open terrain with hills. Show how parameters map to terrain aesthetics.

### Task B: Rule Visualisation Tool
Show before/after states for each possible neighbourhood configuration. Help users understand what each rule does.

### Task C: Basic Pathfinding Analysis
Add a flood-fill or BFS algorithm to check if there is a valid path between two points. Ensure generated levels are playable (connected).

### Task D: Basic Metrics
Calculate and display statistics: open space %, largest connected region, number of isolated caves, evolution of metrics over steps.

### Discussion points
- Show evolution across multiple iterations
- Analyse how parameters affect level design outcomes
- Compare to current research in procedural content generation
- Discuss limitations and possible improvements

In [ ]:
# TODO: Delete this cell

---
## Recommended Reading & Journal Club

### Foundational References

**1. Johnson, L., Yannakakis, G. N. & Togelius, J. (2010)**
*Cellular automata for real-time generation of infinite cave levels.*
Proceedings of the 2010 Workshop on Procedural Content Generation in Games (FDG '10), ACM, 1–4. [DOI](https://doi.org/10.1145/1814256.1814266)
→ The paper that popularised CA cave generation for games; directly relevant to the algorithm implemented in this notebook.

**2. Pech, A., Masek, M., Lam, C.-P. & Hingston, P. (2016)**
*Game level layout generation using evolved cellular automata.*
Connection Science, 28(1), 63–82. [DOI](https://doi.org/10.1080/09540091.2015.1130020)
→ Evolves CA rule sets to generate game levels with desired structural properties.

---

### Journal Club — Procedural Generation with CA

**3. Adams, C. & Louis, S. (2017)**
*Procedural maze level generation with evolutionary cellular automata.*
2017 IEEE Symposium Series on Computational Intelligence (SSCI), IEEE, 1–8. [DOI](https://doi.org/10.1109/SSCI.2017.8285213)
→ Combines evolutionary search with CA to generate maze-like levels with controllable difficulty.

**4. Earle, S., Snider, J., Fontaine, M. C., Nikolaidis, S. & Togelius, J. (2022)**
*Illuminating diverse neural cellular automata for level generation.*
Proceedings of the Genetic and Evolutionary Computation Conference (GECCO '22), ACM, 68–76. [DOI](https://doi.org/10.1145/3512290.3528754)
→ Modern approach using neural CAs trained with quality-diversity algorithms to produce diverse level layouts.